<a href="https://colab.research.google.com/github/gordon921212/Baseball-Project/blob/main/NullModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pybaseball

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.1/426.1 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.7/449.7 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 53.6 MB/s eta 0:00:00


In [2]:
from pybaseball import statcast

df_pitch_by_pitch = statcast(start_dt="2025-03-01", end_dt="2025-11-30")


This is a large query, it may take a moment to complete


/usr/local/lib/python3.12/dist-packages/pybaseball/statcast.py:50: UserWarning: 
That's a nice request you got there. It'd be a shame if something were to happen to it.
We strongly recommend that you enable caching before running this. It's as simple as `pybaseball.cache.enable()`.
Since the Statcast requests can take a *really* long time to run, if something were to happen, like: a disconnect;
gremlins; computer repair by associates of Rudy Giuliani; electromagnetic interference from metal trash cans; etc.;
you could lose a lot of progress. Enabling caching will allow you to immediately recover all the successful
subqueries if that happens.
  warnings.warn(_OVERSIZE_WARNING)


Skipping offseason dates
Skipping offseason dates


  0%|          | 0/246 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
  0%|          | 1/246 [00:05<23:12,  5.68s/it]/usr/local/lib/python3.12/dist-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
  1%|          | 2/246 [00:06<12:31,  3.08s/it]/usr/local/lib/python3.12/dist-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_da

In [3]:
df_in_play= df_pitch_by_pitch[df_pitch_by_pitch['description'] == 'hit_into_play']
df_in_play[['launch_speed','launch_angle','events','hit_distance_sc']]

,launch_speed,launch_angle,events,hit_distance_sc
44,71.9,-24,grounded_into_double_play,5
70,11.9,2,sac_bunt,7
82,89.8,21,double,291
209,93.8,0,field_out,47
296,104.6,39,home_run,366
...,...,...,...,...
3372,<NA>,<NA>,field_out,<NA>
3773,<NA>,<NA>,field_out,<NA>
4763,<NA>,<NA>,field_out,<NA>
2020,<NA>,<NA>,field_out,<NA>


In [4]:
print(df_in_play['events'].unique())
print("各 events 種類的出現次數：")
print(df_in_play['events'].value_counts())

['grounded_into_double_play' 'sac_bunt' 'double' 'field_out' 'home_run'
 'force_out' 'single' 'sac_fly' 'double_play' 'field_error' 'triple'
 'fielders_choice_out' 'fielders_choice' 'sac_fly_double_play'
 'catcher_interf' 'triple_play']
各 events 種類的出現次數：
events
field_out                    79949
single                       28272
double                        8408
home_run                      6070
force_out                     3652
grounded_into_double_play     3405
sac_fly                       1397
field_error                   1112
triple                         690
sac_bunt                       596
double_play                    412
fielders_choice                412
fielders_choice_out            349
sac_fly_double_play             19
catcher_interf                  13
triple_play                      3
Name: count, dtype: int64


#二分類

In [5]:
import pandas as pd

print("開始進行精確的標籤編碼...")

# 1. 剔除雜訊：把「捕手妨礙打擊」的無效數據直接刪掉
df_model = df_in_play[df_in_play['events'] != 'catcher_interf'].copy()

# 2. 定義安打的種類
hit_events = ['single', 'double', 'triple', 'home_run']

# 3. 標籤編碼 (Label Encoding)
# 只要是 hit_events 就是 1，其餘 11 種出局或失誤全部歸為 0
df_model['is_hit'] = df_model['events'].apply(lambda x: 1 if x in hit_events else 0)

# 4. 留下我們需要的欄位 (特徵 X 與目標 Y)
df_model = df_model[['launch_speed', 'launch_angle', 'is_hit']]

# 5. 清除雷達當機沒測到初速/仰角的缺失值
df_model = df_model.dropna()

print("========================================")
print(f"✅ 資料清理完成！共保留 {len(df_model)} 筆有效特徵資料。")
print("========================================")

# 檢查一下 0 和 1 的分佈
print(df_model['is_hit'].value_counts(normalize=True) * 100)

開始進行精確的標籤編碼...
✅ 資料清理完成！共保留 131637 筆有效特徵資料。
is_hit
0    67.77046
1    32.22954
Name: proportion, dtype: float64


In [9]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

# =========================
# 1. Prepare X and y
# =========================

# 定義 X 為 launch_speed 和 launch_angle
# 定義 Y 為 是否打中 (is_hit 在 Data Cleaning 中產生)
X = df_model[['launch_speed', 'launch_angle']]
y = df_model['is_hit']

# 第一次切分：分為 train+validation 和 test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,    # 測試集 佔 20%
    random_state=42,  # 亂數種子
    stratify=y
)

In [10]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, roc_auc_score

print("🥊 [二分類] 準備召喚 Null Model 沙包來挨打...\n")

# ==========================================
# 沙包一號：多數決模型 (永遠猜出局)
# ==========================================
dummy_majority_bin = DummyClassifier(strategy='most_frequent')
# 記得換回你二分類的訓練集資料喔！
dummy_majority_bin.fit(X_train, y_train)

y_pred_majority_bin = dummy_majority_bin.predict(X_test)
y_proba_majority_bin = dummy_majority_bin.predict_proba(X_test)[:, 1] # 二分類抓第1欄(安打機率)

print("========================================")
print("🥊 二分類沙包一號：永遠猜出局 (Most Frequent)")
print("========================================")
auc_majority_bin = roc_auc_score(y_test, y_proba_majority_bin)
print(f"測試集 ROC-AUC: {auc_majority_bin:.4f}\n")
print(classification_report(y_test, y_pred_majority_bin, target_names=['出局 (0)', '安打 (1)'], zero_division=0))


# ==========================================
# 沙包二號：按真實比例瞎猜模型
# ==========================================
dummy_stratified_bin = DummyClassifier(strategy='stratified', random_state=42)
dummy_stratified_bin.fit(X_train, y_train)

y_pred_stratified_bin = dummy_stratified_bin.predict(X_test)
y_proba_stratified_bin = dummy_stratified_bin.predict_proba(X_test)[:, 1]

print("\n========================================")
print("🥊 二分類沙包二號：按比例瞎猜 (Stratified Random)")
print("========================================")
auc_stratified_bin = roc_auc_score(y_test, y_proba_stratified_bin)
print(f"測試集 ROC-AUC: {auc_stratified_bin:.4f}\n")
print(classification_report(y_test, y_pred_stratified_bin, target_names=['出局 (0)', '安打 (1)'], zero_division=0))

🥊 [二分類] 準備召喚 Null Model 沙包來挨打...

🥊 二分類沙包一號：永遠猜出局 (Most Frequent)
測試集 ROC-AUC: 0.5000

              precision    recall  f1-score   support

      出局 (0)       0.68      1.00      0.81     17843
      安打 (1)       0.00      0.00      0.00      8485

    accuracy                           0.68     26328
   macro avg       0.34      0.50      0.40     26328
weighted avg       0.46      0.68      0.55     26328


🥊 二分類沙包二號：按比例瞎猜 (Stratified Random)
測試集 ROC-AUC: 0.4927

              precision    recall  f1-score   support

      出局 (0)       0.67      0.67      0.67     17843
      安打 (1)       0.31      0.31      0.31      8485

    accuracy                           0.56     26328
   macro avg       0.49      0.49      0.49     26328
weighted avg       0.56      0.56      0.56     26328



#三分類

In [11]:
def categorize_hit(event):
    if event == 'single':
        return 1  # 短程安打
    elif event in ['double', 'triple', 'home_run']:
        return 2  # 長打
    else:
        return 0  # 出局 / 非安打

df_multi = df_in_play[['launch_speed', 'launch_angle', 'events']].copy()
df_multi['hit_class'] = df_multi['events'].apply(categorize_hit)

df_multi = df_multi[
    ['launch_speed', 'launch_angle', 'hit_class']
].dropna()

X_multi = df_multi[['launch_speed', 'launch_angle']]
y_multi = df_multi['hit_class']

print("Class distribution:")
print(y_multi.value_counts().sort_index())

Class distribution:
hit_class
0    89222
1    27574
2    14852
Name: count, dtype: int64


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight


X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X_multi,
    y_multi,
    test_size=0.2,
    random_state=42,
    stratify=y_multi
)

In [17]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, roc_auc_score

print("🥊 準備召喚 Null Model 沙包來挨打...\n")
target_names_multi = ['出局 (0)', '一壘安打 (1)', '長打/全壘打 (2)']
# ==========================================
# 沙包一號：多數決模型 (永遠猜出局)
# ==========================================
# strategy='most_frequent' 代表它會去算訓練集裡面誰最多，然後永遠猜那個
dummy_majority = DummyClassifier(strategy='most_frequent')
dummy_majority.fit(X_train_multi, y_train_multi)

# 讓沙包一號去考期末考
y_pred_majority = dummy_majority.predict(X_test_multi)
y_proba_majority = dummy_majority.predict_proba(X_test_multi)

print("========================================")
print("🥊 沙包一號：永遠猜出局 (Most Frequent)")
print("========================================")
# 這裡用 macro 算 AUC
auc_majority = roc_auc_score(y_test_multi, y_proba_majority, multi_class='ovr')
print(f"測試集 ROC-AUC: {auc_majority:.4f}\n")
print(classification_report(y_test_multi, y_pred_majority, target_names=target_names_multi, zero_division=0))


# ==========================================
# 沙包二號：按真實比例瞎猜模型
# ==========================================
# strategy='stratified' 代表它會按照訓練集的比例 (出局多、長打少) 來丟機率骰子
dummy_stratified = DummyClassifier(strategy='stratified', random_state=42)
dummy_stratified.fit(X_train_multi, y_train_multi)

# 讓沙包二號去考期末考
y_pred_stratified = dummy_stratified.predict(X_test_multi)
y_proba_stratified = dummy_stratified.predict_proba(X_test_multi)

print("\n========================================")
print("🥊 沙包二號：按比例瞎猜 (Stratified Random)")
print("========================================")
auc_stratified = roc_auc_score(y_test_multi, y_proba_stratified, multi_class='ovr')
print(f"測試集 ROC-AUC: {auc_stratified:.4f}\n")
print(classification_report(y_test_multi, y_pred_stratified, target_names=target_names_multi, zero_division=0))

🥊 準備召喚 Null Model 沙包來挨打...

🥊 沙包一號：永遠猜出局 (Most Frequent)
測試集 ROC-AUC: 0.5000

              precision    recall  f1-score   support

      出局 (0)       0.68      1.00      0.81     17845
    一壘安打 (1)       0.00      0.00      0.00      5515
  長打/全壘打 (2)       0.00      0.00      0.00      2970

    accuracy                           0.68     26330
   macro avg       0.23      0.33      0.27     26330
weighted avg       0.46      0.68      0.55     26330


🥊 沙包二號：按比例瞎猜 (Stratified Random)
測試集 ROC-AUC: 0.5022

              precision    recall  f1-score   support

      出局 (0)       0.68      0.68      0.68     17845
    一壘安打 (1)       0.21      0.21      0.21      5515
  長打/全壘打 (2)       0.12      0.12      0.12      2970

    accuracy                           0.52     26330
   macro avg       0.34      0.34      0.34     26330
weighted avg       0.52      0.52      0.52     26330

